In [20]:
import pandas as pd

In [21]:
caminho = r'C:\Users\Orçamento\OneDrive - GRUPO RETEC\02. Engenharia\Dep. Orçamentos\POWERBI\AUTOMACAO RD\data\negociacoes_2025.xlsx'

df = pd.read_excel(caminho)


In [22]:
colunas = [
    "id", "name",'Proposta Nº','organization.name','Nome da Obra','Unidade de Negócio','Fator','Local da Obra (Estado)',
    'Local da Obra (Cidade)','Produtos (Representação)',
    'Fábrica (Representação)','Orçamentista', "amount_total", "amount_unique", "markup",
    "created_at", "closed_at", "last_activity_at",
    "interactions", "win", "deal_stage.name", "user.id", "user.name",
    "deal_lost_reason.name","Tipo de Contato", "Fonte de Contato" , "Tipo de Obra", "Produtos (Distribuição)"  
]


In [23]:
df['user.name'].value_counts()

user.name
Luan Araújo         859
Iago Rangel         718
Wellisson Chaves    654
Rutemar Júnior      525
Marlon Souza        423
Gabriel  Bento      176
Bruno Crispim       115
José Nascimento       2
Patrick               1
Ananda Araújo         1
Name: count, dtype: int64

In [24]:
df_dist = df[df['user.name'].isin(['Luan Araújo', 'Iago Rangel','Wellisson Chaves','Rutemar Júnior','Marlon Souza'])]
df_dist = df_dist[colunas].copy()
df_dist.loc[:, "Fator"] = (
    df_dist["Fator"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float, errors="ignore")
)

df_dist['deal_stage.name'].value_counts()

deal_stage.name
Venda Ganha                               2773
Negociação                                 130
Venda Perdida                               93
Conc. Em Andamento                          55
Orç. Enviar (Venda)                         27
Venda Cancelada                             26
Consulta Enviada                            23
Fechado (Aguardando Emissão NF/Pedido)      20
Orç. Enviar (Concorrência)                  11
Orç. Fazendo (Venda)                         6
Orç. Pendente (Venda)                        5
Conc. Perdida                                3
Orç. Pendente (Concorrência)                 2
Consulta                                     2
Aguardando Atendimento                       1
Orç. Fazendo (Concorrência)                  1
Levantamento                                 1
Name: count, dtype: int64

In [25]:
#Definindo os estágios de deal_stage.name que quero pegar
#estágios = ["Venda Ganha", "Venda Perdida", "Venda Cancelada", "Negociação"] #lista com os estágios
#df_dist = df_dist[df_dist["deal_stage.name"].isin(estágios)]

In [26]:
total_orc = len(df_dist["deal_stage.name"] == "Venda Ganha") + len(df_dist["deal_stage.name"] == "Venda Perdida") + len(df_dist["deal_stage.name"] == "Venda Cancelada")
taxa_conversão_rep = df_dist[df_dist["deal_stage.name"] == "Venda Ganha"].shape[0]/total_orc
df_dist['Fator'] = pd.to_numeric(df_dist['Fator'], errors='coerce')
df_dist['Fator'] = df_dist['Fator'].astype(float)

In [27]:
estagios_validos = ['Venda Ganha',"Venda Perdida", "Venda Cancelada"]
df_validos = df_dist[df_dist["deal_stage.name"].isin(estagios_validos)] 
total = df_validos.groupby("organization.name").size()
ganhas = df_validos[df_validos['deal_stage.name']== "Venda Ganha"].groupby('organization.name').size()
taxa_conversao = (ganhas / total)
df_dist["taxa_conversao"] = df["organization.name"].map(taxa_conversao)
df_validos['organization.name'].value_counts()

organization.name
ENGEMIL - ENGENHARIA, EMPREENDIMENTOS, MANUTENCAO E INSTALACAO         71
ARCOBRAS COMERCIO DE AR CONDICIONADO BSB LTDA                          51
WINDSOR ADMINISTRACAO DE HOTEIS E SERVICOS LTDA (ASAL SUL)             45
GOMES ENGENHARIA PROJETOS E CONSULTORIA LTDA                           43
CLIMATECH SOLUCOES EM AR CONDICIONADO LTDA                             39
                                                                       ..
SPE FRANCO RIBEIRO EMPREENDIMENTOS IMOBILIARIOS LTDA                    1
TRINITY EMPREENDIMENTOS DE ENGENHARIA EIRELI                            1
COCENO CONSTRUTORA CENTRO NORTE LTDA                                    1
MENDONCA E GONCALVES CONSTRUCOES E INCORPORACOES LTDA                   1
 AMAZONIA REAL NUTS INDUSTRIA COMERCIO IMPORTACAO E EXPORTACAO LTDA     1
Name: count, Length: 639, dtype: int64

In [28]:
caminho = r"C:\\Users\\Orçamento\\OneDrive - GRUPO RETEC\\02. Engenharia\\Dep. Orçamentos\\POWERBI\\AUTOMACAO RD\\data\\negociacoes_dist_2025.xlsx"
df_dist.to_excel(caminho)